# 12 · Advanced Graphs

> Grafi **pesati**: cammini minimi (Dijkstra, Bellman-Ford), albero ricoprente minimo (Prim), più due classici "costruisci il grafo e poi visitalo" (itinerario euleriano, alfabeto alieno).
> Prerequisito: [11 · Graphs](11%20-%20Graphs.ipynb) · Idee: [Alberi e grafi › Cammini minimi](../Fundamentals/Alberi%20e%20grafi.md#cammini-minimi-con-pesi) · [Albero ricoprente minimo](../Fundamentals/Alberi%20e%20grafi.md#albero-ricoprente-minimo)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Network Delay Time | Dijkstra da una sorgente; risposta = distanza massima | `heapq` di `(dist, nodo)`, `dict` delle distanze |
| Reconstruct Itinerary | cammino euleriano (Hierholzer): usa ogni arco una volta | adiacenza ordinata al contrario + `pop()`, risultato invertito |
| Min Cost to Connect All Points | MST con Prim su grafo completo | `abs(x1 - x2) + abs(y1 - y2)`, heap, `visited` |
| Swim in Rising Water | Dijkstra dove il costo di un percorso è il **massimo** lungo il percorso | `max(cost, grid[nr][nc])` |
| Alien Dictionary | confronta parole adiacenti → archi; poi ordinamento topologico | `zip(w1, w2)`, caso prefisso, Kahn |
| Cheapest Flights Within K Stops | Bellman-Ford limitato a `k + 1` giri | copia dell'array a ogni giro: `prices[:]` |

## Sintassi al volo

| Serve | Sintassi |
|---|---|
| grafo pesato | `graph = defaultdict(list)` · `graph[u].append((v, w))` |
| infinito | `float("inf")`, `math.inf` |
| heap di priorità | `heapq.heappush(heap, (dist, node))` |
| distanza Manhattan | `abs(x1 - x2) + abs(y1 - y2)` |
| ordinare adiacenze | `for u in graph: graph[u].sort(reverse=True)` |
| primo carattere diverso | `for a, b in zip(w1, w2): if a != b: ...; break` |

## Dijkstra

Cammini minimi con pesi **non negativi**. Estrai dall'heap il nodo con distanza minima: quella distanza è definitiva. Le voci "vecchie" rimaste nell'heap si scartano con un controllo.

Costo: `O(E log V)`.

In [1]:
import heapq
from collections import defaultdict

# Rotte tra magazzini: (da, a, minuti)
routes = [(1, 2, 5), (1, 3, 2), (3, 2, 1), (2, 4, 4)]
graph: defaultdict[int, list[tuple[int, int]]] = defaultdict(list)
for u, v, w in routes:
    graph[u].append((v, w))

dist = {1: 0}
heap = [(0, 1)]
while heap:
    d, node = heapq.heappop(heap)
    if d > dist.get(node, float("inf")):
        continue  # voce superata da una distanza migliore
    for nxt, w in graph[node]:
        nd = d + w
        if nd < dist.get(nxt, float("inf")):
            dist[nxt] = nd
            heapq.heappush(heap, (nd, nxt))
print(dist)  # -> {1: 0, 2: 3, 3: 2, 4: 7}

{1: 0, 2: 3, 3: 2, 4: 7}


**Variante "minimax":** se il costo di un percorso non è la somma ma il **massimo** lungo il percorso (es. altezza dell'acqua), cambia solo la riga `nd = d + w` in `nd = max(d, peso)`. Funziona perché anche il massimo non diminuisce mai lungo un percorso.

In [2]:
import heapq

elevation = [[0, 2], [1, 3]]
heap = [(elevation[0][0], 0, 0)]
seen = {(0, 0)}
while heap:
    level, r, c = heapq.heappop(heap)
    if (r, c) == (1, 1):
        print(level)  # -> 3
        break
    for nr, nc in ((r + 1, c), (r, c + 1), (r - 1, c), (r, c - 1)):
        if 0 <= nr < 2 and 0 <= nc < 2 and (nr, nc) not in seen:
            seen.add((nr, nc))
            heapq.heappush(heap, (max(level, elevation[nr][nc]), nr, nc))

3


## Prim: albero ricoprente minimo

Collega tutti i nodi con costo totale minimo. Parti da un nodo, e a ogni passo prendi dall'heap l'arco più economico verso un nodo **non ancora incluso**. Su un grafo completo (ogni punto con ogni altro) non serve costruire gli archi in anticipo: calcoli le distanze quando aggiungi un nodo.

In [3]:
import heapq

points = [(0, 0), (2, 2), (3, 10), (5, 2)]
n = len(points)
in_tree: set[int] = set()
heap = [(0, 0)]  # (costo per collegarlo, nodo)
total = 0
while len(in_tree) < n:
    cost, i = heapq.heappop(heap)
    if i in in_tree:
        continue
    in_tree.add(i)
    total += cost
    xi, yi = points[i]
    for j, (xj, yj) in enumerate(points):
        if j not in in_tree:
            heapq.heappush(heap, (abs(xi - xj) + abs(yi - yj), j))
print(total)  # -> 16 (archi da 4, 3 e 9)

16


## Bellman-Ford con limite di passi

Ogni giro "rilassa" tutti gli archi una volta: dopo `k` giri conosci i cammini minimi che usano **al più `k` archi**. Per limitare i passi, a ogni giro leggi dalle distanze del giro **precedente** (una copia), altrimenti un giro può usare più archi in catena.

In [4]:
flights = [(0, 1, 100), (1, 2, 100), (0, 2, 500)]
n, src, dst, max_stops = 3, 0, 2, 0

prices = [float("inf")] * n
prices[src] = 0
for _ in range(max_stops + 1):        # k scali = k + 1 voli
    previous = prices[:]              # copia: leggi solo il giro precedente
    for u, v, w in flights:
        if previous[u] + w < prices[v]:
            prices[v] = previous[u] + w
print(prices[dst])  # -> 500: senza scali, la rotta da 200 non è ammessa

500


## Hierholzer: usare ogni arco una volta

Per un cammino che usa **tutti gli archi** esattamente una volta: DFS che consuma gli archi (`pop`), e aggiunge il nodo al risultato solo quando non ha più archi in uscita. Il risultato è al contrario.

Per avere l'ordine lessicografico, ordina le adiacenze **al contrario** e usa `pop()` (che toglie dalla fine in `O(1)`) per prendere la più piccola.

In [5]:
from collections import defaultdict

tickets = [("JFK", "SFO"), ("JFK", "ATL"), ("SFO", "ATL"), ("ATL", "JFK"), ("ATL", "SFO")]
graph: defaultdict[str, list[str]] = defaultdict(list)
for src, dst in sorted(tickets, reverse=True):
    graph[src].append(dst)
print(graph["JFK"], graph["JFK"][-1])  # -> ['SFO', 'ATL'] ATL: la più piccola è in fondo

route: list[str] = []
def visit(airport: str) -> None:
    while graph[airport]:
        visit(graph[airport].pop())
    route.append(airport)  # post-ordine

visit("JFK")
print(route[::-1])  # -> ['JFK', 'ATL', 'JFK', 'SFO', 'ATL', 'SFO']

['SFO', 'ATL'] ATL
['JFK', 'ATL', 'JFK', 'SFO', 'ATL', 'SFO']


## Ricavare archi da parole ordinate

Se una lista di parole è ordinata secondo un alfabeto sconosciuto, ogni coppia di parole **adiacenti** dà al più un arco: il primo carattere diverso. Poi applichi l'ordinamento topologico (Kahn, vedi [Graphs](11%20-%20Graphs.ipynb)).

Caso limite: se `w2` è un prefisso proprio di `w1` (`"abc"` prima di `"ab"`), l'ordinamento è impossibile.

In [6]:
words = ["wrt", "wrf", "er", "ett"]
edges = []
for w1, w2 in zip(words, words[1:]):  # coppie adiacenti
    if len(w1) > len(w2) and w1.startswith(w2):
        print("impossibile")
        break
    for a, b in zip(w1, w2):          # zip si ferma alla parola più corta
        if a != b:
            edges.append((a, b))
            break                      # solo il PRIMO carattere diverso conta
print(edges)  # -> [('t', 'f'), ('w', 'e'), ('r', 't')]

all_letters = {ch for word in words for ch in word}
print(sorted(all_letters))  # -> ['e', 'f', 'r', 't', 'w']: anche le lettere senza archi vanno nel risultato

[('t', 'f'), ('w', 'e'), ('r', 't')]
['e', 'f', 'r', 't', 'w']


## Errori comuni

- Dijkstra con pesi negativi: risultato sbagliato. Usa Bellman-Ford.
- Dijkstra senza il controllo `if d > dist[node]: continue`: corretto ma lento.
- Bellman-Ford con limite di passi senza copiare l'array del giro precedente.
- Itinerario: ordinare in modo crescente e poi fare `pop(0)` (`O(n)` per volta) invece di ordinare al contrario e fare `pop()`.
- Alfabeto alieno: dimenticare le lettere che non compaiono in nessun arco, o il caso prefisso.